# 01 — Mega Table Overview

Phase 2 EDA starting point. Goals:
- Row counts and null rates by position
- ADP vs finish correlations
- Feature distributions and outliers
- Prioritize features for Bucket A/B engineering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../../data")

qb = pd.read_csv(DATA_DIR / "training_data_qb.csv")
rb = pd.read_csv(DATA_DIR / "training_data_rb.csv")
wr = pd.read_csv(DATA_DIR / "training_data_wr.csv")
te = pd.read_csv(DATA_DIR / "training_data_te.csv")

tables = {"QB": qb, "RB": rb, "WR": wr, "TE": te}

for pos, df in tables.items():
    print(f"{pos}: {df.shape[0]} rows × {df.shape[1]} cols  |  seasons {df['season'].min()}–{df['season'].max()}")

## Player profile — all seasons for one player

In [ ]:
def player_profile(name: str, pos: str) -> pd.DataFrame:
    df = tables[pos]
    match = df[df["full_name"].str.contains(name, case=False, na=False)]
    display_cols = [
        "full_name", "season", "team",
        "adp_overall", "adp_position_rank", "finish",
        "fantasy_points_ppr",
    ] + [c for c in df.columns if c.startswith("prev_fantasy")]
    present = [c for c in display_cols if c in match.columns]
    return match[present].reset_index(drop=True)

player_profile("Tyreek Hill", "WR")

In [ ]:
# Full feature row for a single player-season — useful for inspecting what the model sees
def feature_row(name: str, pos: str, season: int) -> pd.Series:
    df = tables[pos]
    row = df[(df["full_name"].str.contains(name, case=False, na=False)) & (df["season"] == season)]
    return row.T.rename(columns={row.index[0]: f"{name} {season}"})

feature_row("Tyreek Hill", "WR", 2022)

## Null rates by column

In [ ]:
def null_rates(pos: str) -> pd.DataFrame:
    df = tables[pos]
    pct = (df.isnull().mean() * 100).round(1)
    return pct[pct > 0].rename(f"null% ({pos})").to_frame().sort_values(f"null% ({pos})", ascending=False)

null_rates("WR")

## ADP vs finish — does the market get it right?

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle("ADP position rank vs actual finish (lower = better)", fontsize=13)

for ax, (pos, df) in zip(axes.flat, tables.items()):
    sub = df.dropna(subset=["adp_position_rank", "finish"])
    ax.scatter(sub["adp_position_rank"], sub["finish"], alpha=0.35, s=18, color="steelblue")
    # 45-degree line = perfect prediction
    lim = max(sub["adp_position_rank"].max(), sub["finish"].max()) + 2
    ax.plot([1, lim], [1, lim], "r--", linewidth=1, label="perfect")
    corr = sub["adp_position_rank"].corr(sub["finish"])
    ax.set_title(f"{pos}  (r = {corr:.2f})")
    ax.set_xlabel("ADP position rank")
    ax.set_ylabel("Actual finish")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Correlation of prior-season features with current finish (WR)

In [ ]:
def feature_corr_with_finish(pos: str, top_n: int = 15) -> None:
    df = tables[pos].copy()
    # Use prev_* columns + adp as predictors; finish as outcome
    pred_cols = [c for c in df.columns if c.startswith("prev_") or c in ("adp_overall", "adp_position_rank")]
    sub = df[pred_cols + ["finish"]].dropna(subset=["finish"])
    corr = sub.corr()["finish"].drop("finish").sort_values()

    fig, ax = plt.subplots(figsize=(8, 6))
    colors = ["tomato" if v > 0 else "steelblue" for v in corr.iloc[list(range(top_n)) + list(range(-top_n, 0))]]
    corr.iloc[list(range(top_n)) + list(range(-top_n, 0))].plot(kind="barh", ax=ax, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{pos}: predictor correlation with finish (lower finish = better)")
    ax.set_xlabel("Pearson r")
    plt.tight_layout()
    plt.show()

feature_corr_with_finish("WR")

In [ ]:
# Same chart for every position
for pos in ["QB", "RB", "TE"]:
    feature_corr_with_finish(pos)

## ADP value — biggest over/underperformers vs ADP

In [ ]:
def adp_value(pos: str, n: int = 10) -> None:
    df = tables[pos].dropna(subset=["adp_position_rank", "finish"]).copy()
    # Positive value = finished better than drafted (adp_rank - finish > 0)
    df["value"] = df["adp_position_rank"] - df["finish"]
    label = f"{pos} — top {n} steals and busts"
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print("\nBiggest steals (finished much higher than ADP):")
    print(df.nlargest(n, "value")[["full_name", "season", "adp_position_rank", "finish", "value"]].to_string(index=False))
    print("\nBiggest busts (finished much lower than ADP):")
    print(df.nsmallest(n, "value")[["full_name", "season", "adp_position_rank", "finish", "value"]].to_string(index=False))

adp_value("WR")

In [ ]:
for pos in ["QB", "RB", "TE"]:
    adp_value(pos)